# MCP 클라이언트와 Agent — 실습본

원본: `Chapter02/01-mcp/02_mcp_client.py`  
슬라이드: `99_pdf/Chapter02/day01/3-2-model-context-protocol.pdf`

## 학습 목표

- get_tools와 LLM·ToolNode 연결
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

MCP_CONFIG → get_tools → bind_tools·ToolNode → agent ↔ tools

**읽을 때 확인할 점:** 실습본은 stdio, 정답본은 SSE가 활성화되어 있다. 서버와 클라이언트의 방식을 맞춘다. 슬라이드의 SSE 설정과 비교한다.


## 실행 전 확인

SSE라면 터미널 A에서 서버를 먼저 시작하고 이 셀을 실행한다. stdio라면 클라이언트가 서버 스크립트를 하위 프로세스로 실행한다. LLM 접속도 필요하다.

setup 셀은 `asyncio.run(main())`의 Jupyter 루프 충돌을 피하도록 스크립트 전용 `__main__` 분기를 건너뛰게 합니다. 원본 코드 셀에서 함수를 정의한 뒤, 아래의 별도 `await main()` 셀을 실행합니다. 아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
__file__ = str(_repo_root / "Chapter02/01-mcp/02_mcp_client.py")
# Jupyter에서는 스크립트 전용 __main__ 분기를 실행하지 않습니다.
__name__ = "notebook"


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
"""
MCP 클라이언트 실습

01_mcp_server.py에서 만든 로컬 MCP 서버에 연결해
LangGraph Agent가 도구로 사용하는 예제.

사전 준비:
  터미널 A에서 먼저 실행: python 01_mcp_server.py
  터미널 B에서 이 파일 실행: python 02_mcp_client.py
"""

import asyncio
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()


# ── MCP 서버 연결 설정 ────────────────────────────────────────────────────────
# SSE(Server-Sent Events): HTTP 기반 MCP 연결 방식.
# 01_mcp_server.py를 HTTP 모드로 실행하면 /sse 엔드포인트가 열린다.
# TODO: 2가지 상황에 대해 MCP 연결 방식을 정의한다.
'''
    '서버의 이름' : {
        '실행 방식': sse / stdio
        '요청 경로': 
        '실행 명령어':
        '추가 인자 목록': 
    }
'''

MCP_CONFIG = {
    # 'tutorial': {
    #     'transport': 'sse',
    #     'url': 'http://localhost:8000/sse'
    # },
   'tutorial': {
        "transport": "stdio",
        "command": "python",
        "args": ["Chapter02/01-mcp/01_mcp_server.py", '--stdio'],
    },
}


class State(TypedDict):
    messages: Annotated[list, add_messages]


async def main():
    # TODO: MCP 설정으로 MultiServerMCPClient를 생성해 주세요.
    client = MultiServerMCPClient(MCP_CONFIG)
    # get_tools()는 서버에 연결해 도구 목록을 가져온다.
    # TODO: MCP 서버에서 도구 목록을 가져와 주세요.
        # 서버로부터 정보를 받아와서, 도구화 하고, 내 agent로서도 동작을 해야한다.
        # 해당 작업은 비동기 처리
    tools = await client.get_tools()
    # local_tools = [ ] + [tools]
    print('=== 사용 가능한 MCP 도구 ===')
    for t in tools:
        print(f'  - {t.name}: {t.description}')
    print()

    # TODO: 가져온 MCP 도구를 LLM과 ToolNode에 연결해 주세요.
        # 내 LLM에게 bind_tools를 사용해서 내가 가진 도구 목록을 LLM에게 전달
    llm = get_llm().bind_tools(tools)
        # 그 도구들을 도구 노드로 변환
    tool_node = ToolNode(tools)

    # 우리 그래프에서 해당하는 노드에게 기대하는바
    # 정해진 어떤 LLM을 호출해서
        # 사용자가 보낸 메시지를 토대로
        # 새로운 문자을 만들어 내는 것
    def agent_node(state: State) -> dict:
        # TODO: State의 messages를 도구가 연결된 LLM에 전달해 주세요.
        return {'messages': [llm.invoke(state['messages'])]}

    builder = StateGraph(State)
    builder.add_node('agent', agent_node)
    builder.add_node('tools', tool_node)
    builder.add_edge(START, 'agent')
    # TODO: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.
    # LLM이 도구를 사용해야 한다고 판단한다면 해당 도구를 호출
        # 그게 아니라면 END를...
    builder.add_conditional_edges('agent', tools_condition)
    builder.add_edge('tools', 'agent')

    # TODO: 그래프를 컴파일해 주세요.
    app = builder.compile()

    # 도구를 사용해 요청 처리
    questions = [
        '지금 몇 시야?',
        '체중 70kg, 키 175cm면 BMI가 얼마야?',
        '할 일 목록에 "LangGraph 공부" (우선순위 3)를 추가하고 현재 목록을 보여줘.',
    ]

    for question in questions:
        print(f'Q: {question}')
        result = await app.ainvoke({
            'messages': [HumanMessage(content=question)],
        })
        print(f'A: {result["messages"][-1].content}')
        print()


if __name__ == '__main__':
    asyncio.run(main())


## 클라이언트 실행

서버 연결 조건을 확인한 뒤 다음 셀을 직접 실행하세요. 네트워크 및 LLM 호출이 발생합니다.


In [ ]:
await main()


## CodeMate와 함께 검토

> 활성 MCP_CONFIG를 확인하고 서버 시작 순서와 도구 조회 실패 시 점검 항목을 설명해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. get_tools는 어느 서버에 접속하는가?
2. 도구 실행 후 agent로 되돌아가는 이유는?
